# EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification

**Implementation:** TensorFlow / Keras on Google Colab  
**Dataset used in this notebook:** `tf_flowers` from TensorFlow Datasets  
**Image resolution:** 64 × 64 RGB  
**Split:** 70% training / 15% validation / 15% testing

> Fill in your group number, names and index numbers before submission.

This notebook is designed to satisfy the technical tasks in EN3150 Assignment 03:
- Model A: Standard CNN with Conv2D + MaxPooling.
- Model B: Lightweight CNN using depthwise-separable convolutions, with **< 100,000 trainable parameters**.
- Optimizer comparison: Adam vs SGD vs SGD + Momentum.
- At least 20 epochs for both custom models.
- Test accuracy, confusion matrix, precision and recall.
- Parameter count, model size and training-time comparison.
- Fine-tuning of two lightweight pretrained models: MobileNetV2 and EfficientNetB0.
- Final trade-off comparison focused on accuracy, memory footprint and computational cost.
- Resume-safe training for Google Colab runtime disconnects.

**Important:** Run the notebook yourself and write the report discussion using your own measured results.

## 0. Colab runtime-disconnect strategy

Colab sessions can disconnect. This notebook does not try to bypass Colab's limits. Instead it makes training **recoverable**:

1. Google Drive stores the dataset cache and training artifacts.
2. `BackupAndRestore` saves training state every epoch.
3. `ModelCheckpoint` keeps the best validation model.
4. `CSVLogger` preserves epoch metrics.
5. A custom timing callback saves epoch times to CSV.
6. Completed runs are saved as `final.keras`, so rerunning the training cell will load the completed model instead of starting again.

After a disconnect:
- reconnect to a runtime,
- rerun the setup/import cells,
- remount Drive,
- rerun the dataset/model-definition cells,
- rerun the interrupted training cell.

The training helper will restore from the persistent backup or load the completed model.

In [ ]:
# Optional: install only packages that are not always present in a fresh Colab runtime.
# Do NOT upgrade TensorFlow unnecessarily; Colab already provides a compatible build.
%pip install -q tensorflow-datasets scikit-learn

In [ ]:
import os
import gc
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ----------------------------
# Reproducibility + user fields
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

GROUP_NO = "YOUR_GROUP_NO"
GROUP_MEMBERS = [
    "Name 1 - Index",
    "Name 2 - Index",
    # "Name 3 - Index",
]

IMG_SIZE = 64
BATCH_SIZE = 64
CUSTOM_EPOCHS = 20
OPTIMIZER_COMPARE_EPOCHS = 20
SOTA_HEAD_EPOCHS = 8
SOTA_FINE_TUNE_EPOCHS = 12

print("Group:", GROUP_NO)
print("Members:", GROUP_MEMBERS)

In [ ]:
# -----------------------------------------
# Persistent storage (Google Drive in Colab)
# -----------------------------------------
IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/EN3150_A03")
else:
    PROJECT_ROOT = Path.cwd() / "EN3150_A03"

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
TFDS_ROOT = PROJECT_ROOT / "tfds_data"
RESULTS_ROOT = PROJECT_ROOT / "results"

for p in [ARTIFACT_ROOT, TFDS_ROOT, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Persistent project folder:", PROJECT_ROOT)
print("Artifacts:", ARTIFACT_ROOT)
print("TFDS cache:", TFDS_ROOT)

## 1. Data preparation

We use **TF Flowers**, a 5-class RGB image dataset available through TensorFlow Datasets.  
The original images are resized to **64×64**, meeting the assignment's maximum-resolution requirement.

The same deterministic data partitions are used for all custom and pretrained models:
- Training: 70%
- Validation: 15%
- Testing: 15%

Class labels are:
`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`.

In [ ]:
# The TFDS cache is placed on Drive, so a disconnected runtime does not need
# to download/prepare the full dataset again.

splits = ["train[:70%]", "train[70%:85%]", "train[85%:]"]

(raw_train, raw_val, raw_test), ds_info = tfds.load(
    "tf_flowers",
    split=splits,
    as_supervised=True,
    with_info=True,
    data_dir=str(TFDS_ROOT),
    shuffle_files=False,
)

CLASS_NAMES = ds_info.features["label"].names
NUM_CLASSES = len(CLASS_NAMES)

def cardinality(ds):
    return int(tf.data.experimental.cardinality(ds).numpy())

print("Classes:", CLASS_NAMES)
print("Train:", cardinality(raw_train))
print("Validation:", cardinality(raw_val))
print("Test:", cardinality(raw_test))
print("Total:", cardinality(raw_train) + cardinality(raw_val) + cardinality(raw_test))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], antialias=True)
    image = tf.cast(image, tf.float32)  # Keep 0..255; models handle their own normalization.
    return image, label

train_ds = (
    raw_train
    .shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    raw_test
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Prepared tf.data pipelines.")

In [ ]:
# Visual sanity check
plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(CLASS_NAMES[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Custom architecture design

### Model A — Standard CNN
Architecture:
- Conv2D(32, 3×3, ReLU) → MaxPool
- Conv2D(64, 3×3, ReLU) → MaxPool
- Conv2D(128, 3×3, ReLU) → MaxPool
- Global Average Pooling
- Dense(64, ReLU)
- Dense(number of classes) as logits

### Model B — Lightweight CNN
The same high-level layout is used, but every standard convolution is replaced with `SeparableConv2D`, which performs:
1. a depthwise spatial convolution, then
2. a 1×1 pointwise convolution.

For depth multiplier 1:

**Standard Conv parameters**
\[
(k_h k_w C_{in} + 1)C_{out}
\]

**Depthwise-separable Conv parameters**
\[
k_h k_w C_{in} + C_{in}C_{out} + C_{out}
\]

The hidden activation is **ReLU**, which is hardware-friendly because it is approximately `max(0, x)` and avoids expensive exponentials.  
The output layer returns logits; softmax is not needed just to obtain the predicted class because `argmax(logits)` gives the same class index.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model_a():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1.0 / 255.0)(inputs)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, name="logits")(x)

    return keras.Model(inputs, outputs, name="Model_A_Standard_CNN")


def build_model_b():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1.0 / 255.0)(inputs)

    x = layers.SeparableConv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.SeparableConv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.SeparableConv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, name="logits")(x)

    return keras.Model(inputs, outputs, name="Model_B_Lightweight_CNN")


model_a = build_model_a()
model_b = build_model_b()

model_a.summary()
model_b.summary()

print("Model A parameters:", model_a.count_params())
print("Model B parameters:", model_b.count_params())
assert model_b.count_params() < 100_000, "Model B violates the sub-100k requirement."

In [ ]:
# Manual parameter table for report writing.
def trainable_params(layer):
    return int(sum(np.prod(v.shape) for v in layer.trainable_weights))

def parameter_table(model):
    rows = []
    for layer in model.layers:
        if trainable_params(layer) > 0:
            rows.append({
                "layer": layer.name,
                "type": layer.__class__.__name__,
                "output_shape": str(layer.output.shape),
                "trainable_params": trainable_params(layer),
            })
    return pd.DataFrame(rows)

print("MODEL A")
display(parameter_table(model_a))
print("Total:", model_a.count_params())

print("\nMODEL B")
display(parameter_table(model_b))
print("Total:", model_b.count_params())

In [ ]:
# Approximate MAC count for the custom models.
# Counts Conv2D, SeparableConv2D and Dense MACs; ignores pooling/activation overhead.

def estimate_macs(model):
    total = 0

    for layer in model.layers:
        if isinstance(layer, layers.Conv2D):
            h, w, cout = [int(x) for x in layer.output.shape[1:]]
            cin = int(layer.input.shape[-1])
            kh, kw = layer.kernel_size
            total += h * w * cout * kh * kw * cin

        elif isinstance(layer, layers.SeparableConv2D):
            h, w, cout = [int(x) for x in layer.output.shape[1:]]
            cin = int(layer.input.shape[-1])
            kh, kw = layer.kernel_size
            dm = layer.depth_multiplier
            depthwise = h * w * cin * kh * kw * dm
            pointwise = h * w * (cin * dm) * cout
            total += depthwise + pointwise

        elif isinstance(layer, layers.Dense):
            cin = int(layer.input.shape[-1])
            cout = int(layer.units)
            total += cin * cout

    return int(total)

mac_a = estimate_macs(model_a)
mac_b = estimate_macs(model_b)

print(f"Model A approximate MACs: {mac_a:,}")
print(f"Model B approximate MACs: {mac_b:,}")
print(f"Model B / Model A MAC ratio: {mac_b / mac_a:.3f}")
print(f"Approximate MAC reduction: {(1 - mac_b / mac_a) * 100:.1f}%")

## 3. Resume-safe training utilities

The following helper is central to the Colab-disconnect solution.

Each run gets its own persistent folder:
- `backup/` — optimizer/model/epoch state used by `BackupAndRestore`
- `best.keras` — best validation-accuracy model
- `final.keras` — completed model
- `training_log.csv` — losses and accuracies
- `epoch_times.csv` — training time per epoch

If `final.keras` exists, rerunning the cell loads it immediately.  
If training stopped before completion, Keras restores the last backup and continues.

In [ ]:
LOSS_FN = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

class PersistentEpochTimer(keras.callbacks.Callback):
    def __init__(self, csv_path):
        super().__init__()
        self.csv_path = Path(csv_path)
        self.start_time = None
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)

    def on_epoch_begin(self, epoch, logs=None):
        self.start_time = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.perf_counter() - self.start_time
        row = pd.DataFrame([{"epoch": int(epoch), "seconds": float(elapsed)}])
        header = not self.csv_path.exists()
        row.to_csv(self.csv_path, mode="a", header=header, index=False)


def optimizer_from_name(name):
    name = name.lower()
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=1e-3)
    if name == "sgd":
        return keras.optimizers.SGD(learning_rate=1e-2)
    if name in {"momentum", "sgd_momentum"}:
        return keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9)
    raise ValueError(name)


def load_training_log(run_name):
    path = ARTIFACT_ROOT / run_name / "training_log.csv"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "epoch" in df.columns:
        df = df.drop_duplicates(subset=["epoch"], keep="last").sort_values("epoch")
    return df


def average_epoch_time(run_name):
    path = ARTIFACT_ROOT / run_name / "epoch_times.csv"
    if not path.exists():
        return np.nan
    df = pd.read_csv(path)
    if len(df) == 0:
        return np.nan
    df = df.drop_duplicates(subset=["epoch"], keep="last")
    return float(df["seconds"].mean())


def fit_resumable(model, run_name, optimizer, epochs):
    run_dir = ARTIFACT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    final_path = run_dir / "final.keras"
    best_path = run_dir / "best.keras"
    backup_dir = run_dir / "backup"
    log_path = run_dir / "training_log.csv"
    time_path = run_dir / "epoch_times.csv"

    # If this run already completed, avoid retraining.
    if final_path.exists():
        print(f"[{run_name}] Completed model found. Loading:", final_path)
        loaded = keras.models.load_model(final_path)
        return loaded

    model.compile(
        optimizer=optimizer,
        loss=LOSS_FN,
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

    callbacks = [
        keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_dir),
            save_freq="epoch",
            delete_checkpoint=False,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(log_path), append=True),
        PersistentEpochTimer(time_path),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )

    model.save(final_path)
    print(f"[{run_name}] Saved completed model:", final_path)
    return model


def reset_run(run_name):
    # Use only when you intentionally want to start a run from zero.
    import shutil
    run_dir = ARTIFACT_ROOT / run_name
    if run_dir.exists():
        shutil.rmtree(run_dir)
        print("Deleted:", run_dir)
    else:
        print("Run did not exist:", run_dir)

## 4. Optimizer selection and tuning

Chosen optimizer for the main custom models: **Adam, learning rate = 0.001**.

Comparison:
- Adam: `lr=0.001`
- Standard SGD: `lr=0.01`, momentum = 0
- SGD with Momentum: `lr=0.01`, momentum = 0.9

Why momentum matters:
- Ordinary SGD uses the current gradient only.
- Momentum keeps a velocity term containing part of previous updates.
- This can reduce zig-zagging in steep/narrow loss surfaces and accelerate movement in a consistent descent direction.
- Whether it improves final validation accuracy is an empirical question; use the plots and measured metrics below in your report.

In [ ]:
# Train three independent copies of Model B for optimizer comparison.
# These runs are resumable after Colab disconnects.

model_b_adam = fit_resumable(
    build_model_b(),
    "model_b_adam",
    optimizer_from_name("adam"),
    OPTIMIZER_COMPARE_EPOCHS,
)

model_b_sgd = fit_resumable(
    build_model_b(),
    "model_b_sgd",
    optimizer_from_name("sgd"),
    OPTIMIZER_COMPARE_EPOCHS,
)

model_b_momentum = fit_resumable(
    build_model_b(),
    "model_b_momentum",
    optimizer_from_name("momentum"),
    OPTIMIZER_COMPARE_EPOCHS,
)

In [ ]:
def plot_optimizer_comparison():
    runs = {
        "Adam": "model_b_adam",
        "SGD": "model_b_sgd",
        "SGD + Momentum": "model_b_momentum",
    }

    plt.figure(figsize=(8, 5))
    for label, run in runs.items():
        df = load_training_log(run)
        if not df.empty:
            plt.plot(df["epoch"] + 1, df["val_loss"], label=label)
    plt.xlabel("Epoch")
    plt.ylabel("Validation loss")
    plt.title("Optimizer comparison — validation loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

    plt.figure(figsize=(8, 5))
    for label, run in runs.items():
        df = load_training_log(run)
        if not df.empty:
            plt.plot(df["epoch"] + 1, df["val_accuracy"], label=label)
    plt.xlabel("Epoch")
    plt.ylabel("Validation accuracy")
    plt.title("Optimizer comparison — validation accuracy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

plot_optimizer_comparison()

## 5. Train Model A and use Adam-trained Model B as the main lightweight model

Both custom models are trained for at least 20 epochs.  
The Adam copy already trained in the optimizer experiment is reused as **Model B**, avoiding unnecessary duplicate training.

In [ ]:
model_a_trained = fit_resumable(
    build_model_a(),
    "model_a_adam",
    optimizer_from_name("adam"),
    CUSTOM_EPOCHS,
)

# This is the same completed/resumable Adam run from the optimizer comparison.
model_b_trained = fit_resumable(
    build_model_b(),
    "model_b_adam",
    optimizer_from_name("adam"),
    CUSTOM_EPOCHS,
)

In [ ]:
def plot_custom_training(run_name, title):
    df = load_training_log(run_name)
    if df.empty:
        print("No log found for", run_name)
        return

    plt.figure(figsize=(8, 5))
    plt.plot(df["epoch"] + 1, df["loss"], label="Training loss")
    plt.plot(df["epoch"] + 1, df["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

plot_custom_training("model_a_adam", "Model A — Training vs Validation Loss")
plot_custom_training("model_b_adam", "Model B — Training vs Validation Loss")

## 6. Test evaluation: accuracy, confusion matrix, precision and recall

Macro precision/recall are reported because each class contributes equally to the metric.

In [ ]:
def collect_labels(ds):
    return np.concatenate([y.numpy() for _, y in ds], axis=0)

Y_TEST = collect_labels(test_ds)

def evaluate_classifier(model, name, show_confusion=True):
    logits = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(logits, axis=1)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(Y_TEST, y_pred),
        "precision_macro": precision_score(Y_TEST, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(Y_TEST, y_pred, average="macro", zero_division=0),
    }

    print(pd.Series(metrics))
    print("\nClassification report:")
    print(classification_report(
        Y_TEST,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

    if show_confusion:
        cm = confusion_matrix(Y_TEST, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
        disp.plot(xticks_rotation=45)
        plt.title(f"{name} — Confusion Matrix")
        plt.tight_layout()
        plt.show()

    return metrics

metrics_a = evaluate_classifier(model_a_trained, "Model A")
metrics_b = evaluate_classifier(model_b_trained, "Model B")

In [ ]:
def model_disk_size(path):
    return Path(path).stat().st_size

def estimated_fp32_weight_kb(model):
    # Parameter-only inference weight estimate. Each FP32 value is 4 bytes.
    return model.count_params() * 4 / 1024

custom_comparison = pd.DataFrame([
    {
        "Model": "Model A — Standard CNN",
        "Parameters": model_a_trained.count_params(),
        "Estimated FP32 weight size (KB)": estimated_fp32_weight_kb(model_a_trained),
        "Saved .keras size (KB)": model_disk_size(ARTIFACT_ROOT / "model_a_adam" / "final.keras") / 1024,
        "Avg training time / epoch (s)": average_epoch_time("model_a_adam"),
        "Approx MACs": estimate_macs(model_a_trained),
        "Test accuracy": metrics_a["accuracy"],
    },
    {
        "Model": "Model B — Depthwise Separable",
        "Parameters": model_b_trained.count_params(),
        "Estimated FP32 weight size (KB)": estimated_fp32_weight_kb(model_b_trained),
        "Saved .keras size (KB)": model_disk_size(ARTIFACT_ROOT / "model_b_adam" / "final.keras") / 1024,
        "Avg training time / epoch (s)": average_epoch_time("model_b_adam"),
        "Approx MACs": estimate_macs(model_b_trained),
        "Test accuracy": metrics_b["accuracy"],
    },
])

display(custom_comparison)
custom_comparison.to_csv(RESULTS_ROOT / "custom_model_comparison.csv", index=False)

### What to discuss for Model A vs Model B

Use your measured table, not generic claims.

Points to inspect:
- How much did parameter count decrease?
- How much did the approximate MAC count decrease?
- Did `.keras` file size decrease in a similar proportion?
- Was Model B faster per epoch?
- What accuracy was lost or gained?
- Is the accuracy change worth the memory/compute saving for an edge device?
- Remember that actual deployment speed also depends on hardware/kernel support, not parameter count alone.

## 7. Lightweight SOTA models

We use:
1. **MobileNetV2**
2. **EfficientNetB0**

Both are initialized with ImageNet weights.

Training is performed in two stages:
- Stage 1: freeze the pretrained feature extractor and train the new classifier head for 8 epochs.
- Stage 2: unfreeze the last part of the backbone and fine-tune at a small learning rate for 12 epochs.

Total = 20 epochs/model.

The exact same train/validation/test splits are used.

In [ ]:
def build_mobilenet_v2():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")

    # MobileNetV2 expects values in approximately [-1, 1].
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_rescale")(inputs)

    base = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(NUM_CLASSES, name="logits")(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_TF_Flowers")
    return model, base.name


def build_efficientnet_b0():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")

    # Current Keras EfficientNet includes its own input rescaling preprocessing.
    base = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = False

    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(NUM_CLASSES, name="logits")(x)

    model = keras.Model(inputs, outputs, name="EfficientNetB0_TF_Flowers")
    return model, base.name


def unfreeze_last_layers(model, base_layer_name, n_layers=20):
    base = model.get_layer(base_layer_name)
    base.trainable = True

    # Freeze most of the network.
    for layer in base.layers[:-n_layers]:
        layer.trainable = False

    # Fine-tune last layers, but keep BatchNorm frozen for stable transfer learning.
    for layer in base.layers[-n_layers:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

    return model

In [ ]:
def train_transfer_model(builder, name):
    # -------- Stage 1: classifier head --------
    stage1_name = f"{name}_stage1"
    stage2_name = f"{name}_stage2"

    model, base_name = builder()

    model = fit_resumable(
        model,
        stage1_name,
        keras.optimizers.Adam(learning_rate=1e-3),
        SOTA_HEAD_EPOCHS,
    )

    # -------- Stage 2: fine-tuning --------
    model = unfreeze_last_layers(model, base_name, n_layers=20)

    model = fit_resumable(
        model,
        stage2_name,
        keras.optimizers.Adam(learning_rate=1e-5),
        SOTA_FINE_TUNE_EPOCHS,
    )

    return model


mobilenet_model = train_transfer_model(build_mobilenet_v2, "mobilenet_v2")
efficientnet_model = train_transfer_model(build_efficientnet_b0, "efficientnet_b0")

In [ ]:
metrics_mobile = evaluate_classifier(mobilenet_model, "MobileNetV2")
metrics_eff = evaluate_classifier(efficientnet_model, "EfficientNetB0")

In [ ]:
def benchmark_inference_ms_per_image(model, ds, max_batches=10):
    # Platform-dependent empirical latency; useful as a computational-cost proxy.
    batches = []
    total_images = 0
    for i, (x, _) in enumerate(ds):
        if i >= max_batches:
            break
        batches.append(x)
        total_images += int(x.shape[0])

    # Warm-up
    if batches:
        _ = model(batches[0], training=False)

    start = time.perf_counter()
    for x in batches:
        _ = model(x, training=False)
    elapsed = time.perf_counter() - start

    return (elapsed * 1000.0 / total_images) if total_images else np.nan


final_comparison = pd.DataFrame([
    {
        "Model": "Custom Model B",
        "Parameters": model_b_trained.count_params(),
        "Saved model size (MB)": model_disk_size(ARTIFACT_ROOT / "model_b_adam" / "final.keras") / (1024 ** 2),
        "Test accuracy": metrics_b["accuracy"],
        "Precision (macro)": metrics_b["precision_macro"],
        "Recall (macro)": metrics_b["recall_macro"],
        "Inference ms/image": benchmark_inference_ms_per_image(model_b_trained, test_ds),
    },
    {
        "Model": "MobileNetV2",
        "Parameters": mobilenet_model.count_params(),
        "Saved model size (MB)": model_disk_size(ARTIFACT_ROOT / "mobilenet_v2_stage2" / "final.keras") / (1024 ** 2),
        "Test accuracy": metrics_mobile["accuracy"],
        "Precision (macro)": metrics_mobile["precision_macro"],
        "Recall (macro)": metrics_mobile["recall_macro"],
        "Inference ms/image": benchmark_inference_ms_per_image(mobilenet_model, test_ds),
    },
    {
        "Model": "EfficientNetB0",
        "Parameters": efficientnet_model.count_params(),
        "Saved model size (MB)": model_disk_size(ARTIFACT_ROOT / "efficientnet_b0_stage2" / "final.keras") / (1024 ** 2),
        "Test accuracy": metrics_eff["accuracy"],
        "Precision (macro)": metrics_eff["precision_macro"],
        "Recall (macro)": metrics_eff["recall_macro"],
        "Inference ms/image": benchmark_inference_ms_per_image(efficientnet_model, test_ds),
    },
])

display(final_comparison)
final_comparison.to_csv(RESULTS_ROOT / "final_model_comparison.csv", index=False)

## 8. Final trade-off discussion guide

Do **not** copy a generic conclusion. Base the report on the numbers produced by your run.

### Custom Model B advantages
- Very small parameter count.
- Small memory footprint.
- Much lower custom-layer MAC count than Model A.
- Easier to deploy on highly constrained hardware.
- Training from scratch avoids carrying a large pretrained backbone.

### Custom Model B limitations
- It may have lower accuracy because feature capacity is limited.
- It does not start with rich ImageNet features.
- Performance can be more sensitive to architecture choices and dataset size.

### Pretrained MobileNetV2 / EfficientNetB0 advantages
- Strong pretrained visual features.
- Often higher accuracy with limited task-specific data.
- Mature architectures with extensive deployment support.

### Pretrained-model limitations
- Much larger parameter/memory footprint than a sub-100k custom network.
- More compute and potentially higher latency.
- Fine-tuning is more complex.
- A larger model can exceed the memory/latency budget of a microcontroller-class device.

### Final decision
State which model you would deploy for:
1. a strict microcontroller/very-low-memory target,
2. a Raspberry Pi-class target,
3. a case where accuracy is more important than model size.

Use your measured **accuracy, model size, parameter count and latency** to justify each decision.

## 9. Save compact result summaries

These CSV files are useful when writing the report:
- `custom_model_comparison.csv`
- `final_model_comparison.csv`

They are stored in the persistent Google Drive project folder.

In [ ]:
print("Result files:")
for p in sorted(RESULTS_ROOT.glob("*.csv")):
    print("-", p)

print("\nArtifact runs:")
for p in sorted(ARTIFACT_ROOT.iterdir()):
    if p.is_dir():
        print("-", p.name)

## 10. If Colab disconnects — exact recovery procedure

1. Reconnect to a GPU runtime.
2. Run the package/import cell.
3. Run the reproducibility/config cell.
4. Mount Google Drive again.
5. Run the data-loading/preprocessing cells.
6. Run the model-definition and training-helper cells.
7. Run the training cell that was interrupted.

What happens:
- If the run was incomplete, `BackupAndRestore` restores the latest saved state and continues.
- If the run already completed, `final.keras` is loaded and training is skipped.
- Training logs remain in Drive.

If you intentionally want to retrain one experiment from zero, run:
```python
reset_run("model_b_adam")
```
and then rerun that training cell.

### Avoid these common mistakes
- Do not save large model artifacts inside your GitHub repository.
- Do not store GitHub Personal Access Tokens or API keys inside the notebook.
- Do not rely only on `/content/`; it is deleted when the runtime is reset.
- Do not use browser/JavaScript tricks to fake activity. They do not make your training reproducible.

## 11. GitHub commit plan

The assignment asks you to show that work was done over a reasonable duration.  
Make **real commits as you complete each stage** rather than one final upload.

Suggested progression:
1. `chore: initialize EN3150 assignment repository`
2. `feat: add dataset loading and 70-15-15 preprocessing`
3. `feat: implement standard CNN model A`
4. `feat: implement sub-100k separable CNN model B`
5. `exp: compare Adam SGD and momentum optimizers`
6. `feat: add custom model evaluation and comparison`
7. `feat: add MobileNetV2 transfer learning`
8. `feat: add EfficientNetB0 transfer learning`
9. `analysis: add final accuracy memory and compute comparison`
10. `docs: finalize README and report notes`

Do not fabricate commit dates. Commit when you genuinely complete each part.